In [1]:
import pandas as pd
import numpy as np
from functools import reduce

In [ ]:
class UnweightedMeanMethod:
    """
    This is the simple mean calculation method. When collecting the average mean for any specific species, 
    we only look at information that was given a sample size greater than 0. When given a range of values,
    we are taking the middle value using (max value-min value)/2.
    We than took the arithmetic mean.
    . . . 

    Attributes
    ----------
    data : dataframe
        a dataframe which you want to do calculations with

    Methods
    -------
    __replaceNA(df)
        for the group noted, it replaces all NaN values with 0
    
    getMean(categoryName)
        returns the mean of all fruit information of the selected category
    """
    #Reads in and initilizes the dataframe for the class
    def __init__(self, data):
        self.df = pd.read_csv(data)

 #This method
    def __replaceNA(self,df):

        #Gets the last, sample size columns, and replaces any unknown values with 0 
        last_column_name = df.columns[-1]
        df.fillna({last_column_name: 0},inplace=True)
        df[last_column_name] = df[last_column_name].fillna(0)
        
        #Gets the column index for the upper and lower limit for the range 
        index_upper_column = next((i for i, col in enumerate(df.columns) if "upper" in col), None)
        index_lower_column = next((i for i, col in enumerate(df.columns) if "lower" in col), None)

        #Finds the average of the range for each row and fills in the trait mean coloumn
        Num_average = df.iloc[:, [index_upper_column, index_lower_column]].mean(axis=1)

        #Converts the category column to float, removing any non-numeric characters
        category_name = df.columns[1]
        df[category_name] = df[category_name].replace(r'[a-zA-Z<>+=]', '', regex=True).astype(float)

        #Takes the category and replaces the NaN with the mean calculated from the range does not touch cells with non-NaN values
        df.fillna({category_name: Num_average},inplace=True)
        df[category_name] = df[category_name].fillna(Num_average)
        
        #Returns the dataframe with filled in means and fixed sample size
        return df
    
    #This method takes the mean of the selected catagory
    def getMean(self,categoryName):
        #Only includes the rows that are being used in the study
        only_completed = self.df[self.df['in_publication']==1.0]

        #Get the relevent information and combine them
        plant = only_completed[["plant_species"]]
        infoDf = only_completed.iloc[:,only_completed.columns.get_loc(categoryName):only_completed.columns.get_loc(categoryName)+6]
        df =  pd.concat([plant, infoDf], axis=1)

        #Make the data in desired formatting
        df = self.__replaceNA(df)
        
        #Get the mean calculation and remove the NaNs (species that doesn't have any data associated with it)
        df = df.groupby(["plant_species"])[categoryName].mean().reset_index()
        
        #Drop all rows that doesn't have a mean (is NaN)
        df.dropna(inplace=True)

        return df

df = pd.read_csv("supplement_database.csv")
p1= UnweightedMeanMethod("supplement_database.csv")

numberOfSeeds1 = p1.getMean("number_s_per_f")
seedWidth1 = p1.getMean("s_width")
fruitWidth1 = p1.getMean("f_width")

numberOfSeeds1.to_csv("numberOfSeedsMethod1.csv", index=False)
seedWidth1.to_csv("seedWidthMethod1.csv", index=False)
fruitWidth1.to_csv("fruitWidthMethod1.csv", index=False)

In [ ]:
class WeightedMeanAllDataMethod:
    """
    This is the first out of two methods to test to see if there is a better 
    alternative to averaging that would result in a more accurate as well as a more statstically sound 
    analysis. For this method, all previous conditions in method 1 are continued. However, 
    instead of taking the arithemtic mean, we take the weighted mean 
    where the weights are the sample size, and unknown sample size is given a weight of 1. 

    Attributes
    ----------
    data : dataframe
        a dataframe which you want to do calculations with

    Methods
    -------
    __replaceNA(df)
        for the group noted, it replaces all NaN values with 0
    
    getMean(categoryName)
        returns the mean of all fruit information of the selected category
    """
    #reads in and initilizes the dataframe for the class
    def __init__(self, data):
        self.df = pd.read_csv(data)
    
    #this method
    def __replaceNA(self,df):
        #gets the column index for the upper and lower limit for the range 
        index_upper_column = next((i for i, col in enumerate(df.columns) if "upper" in col), None)
        index_lower_column = next((i for i, col in enumerate(df.columns) if "lower" in col), None)

        #finds the average of the range for each row and fills in the trait mean coloumn
        Num_average = df.iloc[:, [index_upper_column, index_lower_column]].mean(axis=1)
        
        # Converts the category column to float, removing any non-numeric characters
        category_name = df.columns[1]
        df[category_name] = df[category_name].replace(r'[a-zA-Z<>+=]', '', regex=True).astype(float)

        #takes the category and replaces the NaN with the mean calculated from the range does not touch cells with non-NaN values
        df.fillna({category_name: Num_average},inplace=True)
        df[category_name] = df[category_name].fillna(Num_average)
        
        #returns the dataframe with filled in means and fixed sample size
        return df
    
    def weighted_average(self, values, weights):
        """
        Computes a weighted average for the given values and weights.

        Parameters:
        values (pd.Series): A pandas Series containing the values to average.
        weights (pd.Series): A pandas Series containing the corresponding weights.

        Returns:
        float: The weighted average of the valid (non-NaN) values using the given weights, or NaN if all values are NaN.
        """
    
        # Remove NaN values from 'values' and get the corresponding valid weights.
        valid_values = values.dropna()  # Keeps only non-NaN values in the 'values' series.
        valid_weights = weights.loc[valid_values.index]  # Select weights that correspond to the valid indices.

        # Check if there are any valid values to compute the weighted average.
        if not valid_values.empty:
            # Compute and return the weighted average using numpy.
            return np.average(valid_values, weights=valid_weights)
    
        # If no valid values are present, return NaN.
        return np.nan

    #This method takes the mean of the selected catagory
    def getMean(self,categoryName):
        #Only includes the rows that are being used in the study
        only_completed = self.df[self.df['in_publication']==1.0]

        #Get the relevent information and combine them
        plant = only_completed[["plant_species"]]
        infoDf = only_completed.iloc[:,only_completed.columns.get_loc(categoryName):only_completed.columns.get_loc(categoryName)+6]
        df =  pd.concat([plant, infoDf], axis=1)
        
        #Make the data in desired formatting
        df = self.__replaceNA(df)

        #Replace all sample sizes reported as "0" and convert them to a "1"
        sample_columns = df.columns[-1]
        df[sample_columns] = df[sample_columns].replace(0,1)
    
        # Calculate the weighted mean for non-NaN values in each group
        weighted_means = (df.groupby('plant_species').apply(lambda group: pd.Series({'plant_species': group.name,categoryName: self.weighted_average(group[categoryName], group[sample_columns])}), include_groups=False).reset_index(drop=True))
        
        #Drop all rows that doesn't have a mean (is NaN)
        weighted_means.dropna(inplace=True)
        
        return weighted_means

df = pd.read_csv("supplement_database.csv")
p2= WeightedMeanAllDataMethod("supplement_database.csv")

numberOfSeeds2 = p2.getMean("number_s_per_f")
seedWidth2 = p2.getMean("s_width")
fruitWidth2 = p2.getMean("f_width")

numberOfSeeds2.to_csv("numberOfSeedsMethod2.csv", index=False)
seedWidth2.to_csv("seedWidthMethod2.csv", index=False)
fruitWidth2.to_csv("fruitWidthMethod2.csv", index=False)

In [ ]:
class WeightedMeanExcludingDataWithoutSampleSizeMethod:
    """
    This is the third and most restrictive out of three methods as it only takes into account samples where we were given a sample size 
    For the remaining we should take the weighted means per each fruit species group.
    . . . 

    Attributes
    ----------
    data : dataframe
        a dataframe which you want to do calculations with

    Methods
    -------
    __replaceNA(df)
        for the group noted, it replaces all NaN values with 0
    
    getMean(categoryName)
        returns the mean of all fruit information of the selected category
    """

    #Reads in and initilizes the dataframe for the class
    def __init__(self, data):
        self.df = pd.read_csv(data)
    
    #This method cleans up the data before any further analysis can be done
    def __replaceNA(self,df):

        #Gets the last, sample size columns, and replaces any 0 with NaN
        last_column_name = df.columns[-1]
        df[last_column_name]= df[last_column_name].replace(0,np.nan)
        
        #Gets the column index for the upper and lower limit for the range 
        index_upper_column = next((i for i, col in enumerate(df.columns) if "upper" in col), None)
        index_lower_column = next((i for i, col in enumerate(df.columns) if "lower" in col), None)

        #Finds the average of the range for each row and fills in the trait mean coloumn
        Num_average = df.iloc[:, [index_upper_column, index_lower_column]].mean(axis=1)

        #Converts the category column to float, removing any non-numeric characters
        category_name = df.columns[1]
        df[category_name] = df[category_name].replace(r'[a-zA-Z<>+=]', '', regex=True).astype(float)

        #Takes the category and replaces the NaN with the mean calculated from the range does not touch cells with non-NaN values
        df.fillna({category_name: Num_average},inplace=True)
        df[category_name] = df[category_name].fillna(Num_average)

        #Returns the dataframe with filled in means and fixed sample size
        return df
    

    def weighted_average_ignore_nan(self, values, weights):
        """
        Calculate the weighted average of `values`, ignoring NaN weights.

        Parameters:
        - values (array-like): The values to be averaged.
        - weights (array-like): The corresponding weights for the values.

        Returns:
        - float: The weighted average, or NaN if all weights are NaN.
        """
    #Check if all weights are NaN. If so, return NaN as the result.
        if np.all(np.isnan(weights)):
            return np.nan
    
    #Calculate the weighted average ignoring NaN values:
        return np.sum(values * weights) / np.sum(weights)

    
    #This method takes the mean of the selected catagory
    def getMean(self,categoryName):
        
        #Only includes the rows that are being used in the study
        only_completed = self.df[self.df['in_publication']==1.0]

        #Get the relevent information and combine them
        plant = only_completed[["plant_species"]]
        infoDf = only_completed.iloc[:,only_completed.columns.get_loc(categoryName):only_completed.columns.get_loc(categoryName)+6]
        df =  pd.concat([plant, infoDf], axis=1)

        #Make the data in desired formatting
        df = self.__replaceNA(df)
        print(df.isna().sum())
        print(df.shape[0])

        #Remove all rows where we were not given a sample size when it was reported
        sample_columns = df.columns[-1]
        df[sample_columns] = df[sample_columns].replace(0,np.nan)
        df.dropna(subset=[sample_columns], inplace=True)
        
        #Calculates the weighted mean for each group defined by the 'Plant Species' column, where the weights are given by the values in the 'sample_columns' column, and the mean is calculated for the values in the 'categoryName' column.
        mean = (df.groupby('plant_species').apply(lambda group: pd.Series({'plant_species': group.name,categoryName: self.weighted_average_ignore_nan(group[categoryName], group[sample_columns])}), include_groups=False).reset_index(drop=True))
        
        #Join everything together
        new_column_names = ['plant_species', categoryName]
        mean.columns = new_column_names
        
        #Return the dataframe
        return mean
    
df = pd.read_csv("supplement_database.csv")
p3= WeightedMeanExcludingDataWithoutSampleSizeMethod("supplement_database.csv")

numberOfSeeds3 = p3.getMean("number_s_per_f")
seedWidth3 = p3.getMean("s_width")
fruitWidth3 = p3.getMean("f_width")

numberOfSeeds3.to_csv("numberOfSeedsMethod3.csv", index=False)
seedWidth3.to_csv("seedWidthMethod3.csv", index=False)
fruitWidth3.to_csv("fruitWidthMethod3.csv", index=False)